In [2]:
import random
import time

import tensorflow as tf
from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix,  precision_score, recall_score, f1_score, accuracy_score,roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler,  LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd


In [3]:
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/KDDCUP99/train_kdd.csv",compression="infer")
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/KDDCUP99/test_kdd.csv", compression= "infer")

In [4]:
train_df['outcome'] = train_df['outcome'].str.rstrip('.')
test_df['outcome'] = test_df['outcome'].str.rstrip('.')

train_df.rename(columns={"outcome": "label"}, inplace=True)
test_df.rename(columns={"outcome": "label"}, inplace=True)

In [5]:
train_df["binary_label"] = train_df["label"].where(
    train_df["label"].eq("normal"),
    "attack"
)
test_df["binary_label"] = test_df["label"].where(
  test_df["label"].eq("normal"),
    "attack"
)

In [6]:
darpa_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/Darpa/Darpa_Pcap_KDD_features.csv")
unsw_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/UNSW-NB15/UNSW_Pcap_NSL_features.csv")
cic2017_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2017/CIC_Pcap_NSL_features.csv")
cic2018_pcap = pd.read_csv("/content/drive/MyDrive/Array_submission/Dataset/Darpa-features/CIC2018/CIC18_Pcap_KDD_features.csv")

In [7]:
train_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'binary_label'],
      dtype='object')

In [8]:
darpa_pcap.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty_level',
       'attack_family'],
      dtype='object')

In [9]:
cic2017_pcap.rename(columns={"category": "label"}, inplace=True)
unsw_pcap.drop(columns=["label"], inplace=True)
unsw_pcap.rename(columns={"attack_type": "label"}, inplace=True)
darpa_pcap.drop(columns=["label", "difficulty_level"], inplace=True)
darpa_pcap.rename(columns={"attack_family": "label"}, inplace=True)

In [10]:
darpa_pcap['source_pcap'] = "darpa"
unsw_pcap['source_pcap'] = "unsw-nb15"
cic2017_pcap['source_pcap'] = "cic2017"
cic2018_pcap['source_pcap'] = "cic2018"

In [11]:
data_list = [darpa_pcap,unsw_pcap,cic2017_pcap,cic2018_pcap]
df = pd.concat(data_list)
df.shape

(2884849, 44)

In [ ]:
import re

LABEL_COLUMN = "label"

df["original_label"] = df[LABEL_COLUMN].copy()

cleaned_labels = (
    df[LABEL_COLUMN]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s*-\s*attempted\s*$",
        "",
        regex=True,
        flags=re.IGNORECASE,
    )
    .str.strip()
)

normalized_labels = cleaned_labels.str.casefold()

df[LABEL_COLUMN] = cleaned_labels

benign_mask = normalized_labels.isin(["normal", "benign"])
df.loc[benign_mask, LABEL_COLUMN] = "benign"

recon_mask = normalized_labels.isin(["probe", "reconnaissance"])
df.loc[recon_mask, LABEL_COLUMN] = "reconnaissance"



,count
label,
benign,2651612
dos,210758
reconnaissance,10513
Exploits,5371
Fuzzers,3760
Web Attack - Brute Force,1111
Generic,675
Web Attack - XSS,587
Shellcode,146


In [ ]:
dos_mask = cleaned_labels.str.match(
    r"^(dos\b|dos[_\s-])",
    case=False,
    na=False,
)
df.loc[dos_mask, LABEL_COLUMN] = "dos"

display(df[LABEL_COLUMN].value_counts(dropna=False).to_frame("count"))

In [ ]:
source_label_counts = (
    df.groupby(["source_pcap", "label"], dropna=False)
      .size()
      .reset_index(name="count")
)

source_totals = (
    source_label_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

source_label_counts["within_source_pct"] = (
    100 * source_label_counts["count"] / source_totals
)

display(
    source_label_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,label,count,within_source_pct
5,cic2017,benign,661240,80.345567
6,cic2017,dos,160023,19.443982
2,cic2017,Web Attack - Brute Force,1111,0.134995
4,cic2017,Web Attack - XSS,587,0.071325
1,cic2017,Infiltration,21,0.002552
3,cic2017,Web Attack - Sql Injection,12,0.001458
0,cic2017,Heartbleed,1,0.000122
7,cic2018,benign,661240,94.944088
8,cic2018,dos,35202,5.054476
9,cic2018,infiltration,10,0.001436


In [14]:
protocol_counts = (
    df.groupby(["source_pcap", "protocol_type"], dropna=False)
      .size()
      .reset_index(name="count")
)

protocol_totals = (
    protocol_counts.groupby("source_pcap")["count"]
    .transform("sum")
)

protocol_counts["within_source_pct"] = (
    100 * protocol_counts["count"] / protocol_totals
)

display(
    protocol_counts.sort_values(
        ["source_pcap", "count"],
        ascending=[True, False]
    )
)

,source_pcap,protocol_type,count,within_source_pct
1,cic2017,tcp,424414,51.569451
2,cic2017,udp,398279,48.393854
0,cic2017,icmp,302,0.036695
4,cic2018,tcp,679248,97.529765
5,cic2018,udp,9181,1.318253
3,cic2018,icmp,8023,1.151982
7,darpa,tcp,833984,77.316957
8,darpa,udp,240635,22.308781
6,darpa,icmp,4037,0.374262
10,unsw-nb15,tcp,212939,74.260495


In [15]:
source_label_protocol = (
    df.groupby(
        ["source_pcap", "label", "protocol_type"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

display(
    source_label_protocol.sort_values(
        ["source_pcap", "label", "count"],
        ascending=[True, True, False]
    )
)

,source_pcap,label,protocol_type,count
0,cic2017,Heartbleed,tcp,1
1,cic2017,Infiltration,tcp,21
2,cic2017,Web Attack - Brute Force,tcp,1111
3,cic2017,Web Attack - Sql Injection,tcp,12
4,cic2017,Web Attack - XSS,tcp,587
7,cic2017,benign,udp,398279
6,cic2017,benign,tcp,262660
5,cic2017,benign,icmp,301
9,cic2017,dos,tcp,160022
8,cic2017,dos,icmp,1


In [ ]:
# Overall source share
source_share = (
    df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("overall_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

attack_df = df[df["label"] != "benign"]

attack_source_share = (
    attack_df["source_pcap"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("attack_share_pct")
    .reset_index()
    .rename(columns={"index": "source_pcap"})
)

display(source_share)
display(attack_source_share)

,source_pcap,overall_share_pct
0,darpa,37.390380
1,cic2017,28.528183
2,cic2018,24.141714
3,unsw-nb15,9.939723


,source_pcap,attack_share_pct
0,cic2017,69.352204
1,cic2018,15.097090
2,darpa,10.564790
3,unsw-nb15,4.985916


In [ ]:
df_master = df.copy()

df_master["binary_label"] = df_master["label"].where(
    df_master["label"].eq("benign"),
    "attack"
)

matched_category_sources = {
    "dos": {"darpa", "unsw-nb15", "cic2017", "cic2018"},
    "reconnaissance": {"darpa", "unsw-nb15"},
}

def assign_category_scope(row):
    label = row["label"]
    source = row["source_pcap"]

    if label == "benign":
        return "benign"

    valid_sources = matched_category_sources.get(label)

    if valid_sources is not None and source in valid_sources:
        return "matched_category"

    return "source_specific_or_unmatched"

df_master["category_scope"] = df_master.apply(
    assign_category_scope,
    axis=1
)

df_master["feature_schema"] = "kdd_nsl"

scope_summary = (
    df_master
    .groupby(
        ["source_pcap", "label", "binary_label", "category_scope"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["source_pcap", "category_scope", "count"],
        ascending=[True, True, False]
    )
)

display(scope_summary)

,source_pcap,label,binary_label,category_scope,count
5,cic2017,benign,benign,benign,661240
6,cic2017,dos,attack,matched_category,160023
2,cic2017,Web Attack - Brute Force,attack,source_specific_or_unmatched,1111
4,cic2017,Web Attack - XSS,attack,source_specific_or_unmatched,587
1,cic2017,Infiltration,attack,source_specific_or_unmatched,21
3,cic2017,Web Attack - Sql Injection,attack,source_specific_or_unmatched,12
0,cic2017,Heartbleed,attack,source_specific_or_unmatched,1
7,cic2018,benign,benign,benign,661240
8,cic2018,dos,attack,matched_category,35202
9,cic2018,infiltration,attack,source_specific_or_unmatched,10


In [ ]:

from pathlib import Path
import gc
import random
import time
import warnings

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_recall_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.utils.validation import check_is_fitted

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    LSTM,
)

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "The voting ensemble requires xgboost. "
        "Install it with: pip install xgboost"
    ) from exc


FEATURE_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login",
    "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]

CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]
NUMERICAL_COLUMNS = [
    column for column in FEATURE_COLUMNS
    if column not in CATEGORICAL_COLUMNS
]

LABEL_COLUMN = "label"
SOURCE_COLUMN = "source_pcap"

SEEDS = [11, 22, 33, 44, 55]
EXTERNAL_TEST_SIZE = 0.30
PREPARATION_SEED = 42
VALIDATION_SIZE = 0.20

MODEL_KEYS = [
    "xiao_ae_cnn",
    "jia_dnn",
    "anuja_ae_cnn_lstm",
    "smart_city_voting_ensemble",
]

MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

EPOCHS = {
    "xiao_autoencoder": 20,
    "xiao_cnn": 50,
    "jia_dnn": 20,
    "anuja_autoencoder": 10,
    "anuja_classifier": 10,
}

BATCH_SIZES = {
    "xiao": 128,
    "jia": 128,
    "anuja": 128,
}

TARGET_TEST_NORMAL_FRACTION = 0.195
TARGET_TEST_ATTACK_FRACTION = 1.0 - TARGET_TEST_NORMAL_FRACTION
EXP4_EXPERIMENT = "exp4_mixed_traffic_test_ratio19p5_normal"

SAVE_RATIO_MATCHED_TEST_SETS = False

RESULTS_DIR = Path("kdd99_exp4_ratio19p5_normal_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR.resolve())


Results directory: /content/kdd99_exp4_ratio19p5_normal_results


In [ ]:

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def normalize_label(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().rstrip(".").casefold()


def binary_y(frame):
    labels = frame[LABEL_COLUMN].map(normalize_label)
    return np.where(
        labels.isin(["normal", "benign"]),
        0,
        1,
    ).astype("int8")


def validate_frame(frame, name):
    missing = sorted(
        set(FEATURE_COLUMNS + [LABEL_COLUMN])
        - set(frame.columns)
    )
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")


def optional_sample(frame, maximum, seed):
    if maximum is None or len(frame) <= maximum:
        return frame.reset_index(drop=True)

    sampled, _ = train_test_split(
        frame,
        train_size=maximum,
        random_state=seed,
        stratify=binary_y(frame),
    )
    return sampled.reset_index(drop=True)


def prepare_test(frame, seed):
    return optional_sample(frame, MAX_TEST_ROWS, seed)


validate_frame(train_df, "KDD99 training data")
validate_frame(test_df, "KDD99 test data")
validate_frame(df_master, "heterogeneous pool")

print("KDD99 train:", train_df.shape)
print("KDD99 test:", test_df.shape)
print("External pool:", df_master.shape)


KDD99 train: (494021, 43)
KDD99 test: (311029, 43)
External pool: (2884849, 48)


In [ ]:

SHARED_CATEGORIES_BY_SOURCE = {
    "darpa": {"benign", "dos", "reconnaissance"},
    "unsw-nb15": {"benign", "dos", "reconnaissance"},
    "cic2017": {"benign", "dos"},
    "cic2018": {"benign", "dos"},
}


def shared_external_pool(master):
    parts = []
    for source, categories in SHARED_CATEGORIES_BY_SOURCE.items():
        part = master[
            master[SOURCE_COLUMN].eq(source)
            & master[LABEL_COLUMN].isin(categories)
        ].copy()
        parts.append(part)

    result = pd.concat(parts, ignore_index=True)
    return result


def split_each_source(frame, stratify_column, test_size, seed):
    train_parts = []
    test_parts = []

    for source, source_frame in frame.groupby(
        SOURCE_COLUMN,
        sort=True,
    ):
        source_frame = source_frame.reset_index(drop=True)

        train_part, test_part = train_test_split(
            source_frame,
            test_size=test_size,
            random_state=seed,
            shuffle=True,
            stratify=source_frame[stratify_column],
        )

        train_parts.append(train_part)
        test_parts.append(test_part)

    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
    )


def balance_shared_external_training(frame, seed):
    rng = np.random.default_rng(seed)
    selected_parts = []

    attack_labels = [
        label
        for label in ["dos", "reconnaissance"]
        if label in set(frame[LABEL_COLUMN])
    ]

    attack_parts_by_source = {
        source: []
        for source in frame[SOURCE_COLUMN].unique()
    }

    for label in attack_labels:
        label_frame = frame[frame[LABEL_COLUMN].eq(label)]
        source_counts = label_frame.groupby(SOURCE_COLUMN).size()

        if len(source_counts) < 2:
            continue

        quota = int(source_counts.min())

        for source, source_frame in label_frame.groupby(SOURCE_COLUMN):
            chosen = source_frame.sample(
                n=quota,
                random_state=seed,
                replace=False,
            )
            attack_parts_by_source[source].append(chosen)
            selected_parts.append(chosen)

    for source, attack_parts in attack_parts_by_source.items():
        if not attack_parts:
            continue

        source_attack_count = sum(len(part) for part in attack_parts)
        benign_frame = frame[
            frame[SOURCE_COLUMN].eq(source)
            & frame[LABEL_COLUMN].eq("benign")
        ]

        benign_count = min(source_attack_count, len(benign_frame))
        selected_parts.append(
            benign_frame.sample(
                n=benign_count,
                random_state=seed,
                replace=False,
            )
        )

    if not selected_parts:
        raise ValueError("No rows were selected for shared balancing.")

    result = pd.concat(selected_parts, ignore_index=True)
    return result.sample(
        frac=1,
        random_state=seed,
    ).reset_index(drop=True)


def balance_binary_external_training(frame, seed):
    working = frame.copy()
    working["_binary"] = binary_y(working)

    counts = (
        working
        .groupby([SOURCE_COLUMN, "_binary"])
        .size()
    )

    quota = int(counts.min())
    selected = []

    for (source, binary_class), group in working.groupby(
        [SOURCE_COLUMN, "_binary"]
    ):
        selected.append(
            group.sample(
                n=quota,
                random_state=seed,
                replace=False,
            )
        )

    result = pd.concat(selected, ignore_index=True)
    return (
        result
        .drop(columns="_binary")
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


def show_distribution(frame, title):
    table = (
        frame
        .assign(_binary=binary_y(frame))
        .groupby([SOURCE_COLUMN, LABEL_COLUMN, "_binary"])
        .size()
        .reset_index(name="count")
    )
    print(title, frame.shape)
    display(table)


def _largest_remainder_allocation(counts, requested_total):
    counts = pd.Series(counts, dtype="int64")
    available_total = int(counts.sum())

    if requested_total < 0 or requested_total > available_total:
        raise ValueError(
            f"requested_total={requested_total} exceeds "
            f"available_total={available_total}"
        )

    if requested_total == 0:
        return counts * 0

    raw = counts / available_total * requested_total
    allocation = np.floor(raw).astype("int64")
    remaining = int(requested_total - allocation.sum())
    fractional = (raw - allocation).sort_values(ascending=False)

    while remaining > 0:
        progressed = False
        for stratum in fractional.index:
            if allocation.loc[stratum] < counts.loc[stratum]:
                allocation.loc[stratum] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError("Could not complete proportional allocation.")

    return allocation


def proportional_stratified_sample(
    frame,
    requested_total,
    strata_column,
    random_state,
):
    """
    Sample without replacement while preserving the stratum distribution as
    closely as integer counts permit.
    """
    if requested_total == len(frame):
        return frame.copy()

    strata = (
        frame[strata_column]
        .astype("string")
        .fillna("<missing>")
    )
    counts = strata.value_counts(sort=False)
    allocation = _largest_remainder_allocation(
        counts,
        requested_total,
    )

    sampled_parts = []
    ordered_strata = sorted(allocation.index.astype(str))

    for offset, stratum in enumerate(ordered_strata):
        take = int(allocation.loc[stratum])
        if take == 0:
            continue

        cell = frame.loc[strata.eq(stratum)]
        sampled_parts.append(
            cell.sample(
                n=take,
                replace=False,
                random_state=random_state + offset,
            )
        )

    if not sampled_parts:
        return frame.iloc[0:0].copy()

    return pd.concat(sampled_parts, axis=0)


def match_binary_test_ratio(
    frame,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
):

    if not 0.0 < target_attack_fraction < 1.0:
        raise ValueError("target_attack_fraction must be between 0 and 1.")

    working = frame.copy()
    working["__ratio_original_order"] = np.arange(len(working))
    working["__ratio_binary"] = binary_y(working)

    attack_rows = working[working["__ratio_binary"].eq(1)]
    normal_rows = working[working["__ratio_binary"].eq(0)]

    available_attack = int(len(attack_rows))
    available_normal = int(len(normal_rows))

    if available_attack == 0 or available_normal == 0:
        raise ValueError(
            "Every ratio-matched test source must contain both attack "
            "and normal rows."
        )

    target_normal_fraction = 1.0 - target_attack_fraction
    maximum_total = min(
        int(np.floor(available_attack / target_attack_fraction)),
        int(np.floor(available_normal / target_normal_fraction)),
    )

    while maximum_total > 1:
        selected_attack = int(
            round(maximum_total * target_attack_fraction)
        )
        selected_normal = maximum_total - selected_attack

        if (
            0 < selected_attack <= available_attack
            and 0 < selected_normal <= available_normal
        ):
            break

        maximum_total -= 1
    else:
        raise ValueError("Not enough rows to construct a matched test set.")

    sampled_attack = proportional_stratified_sample(
        attack_rows,
        selected_attack,
        LABEL_COLUMN,
        random_state,
    )
    sampled_normal = normal_rows.sample(
        n=selected_normal,
        replace=False,
        random_state=random_state + 10_000,
    )

    matched = (
        pd.concat([sampled_attack, sampled_normal], axis=0)
        .sort_values("__ratio_original_order")
        .drop(
            columns=[
                "__ratio_original_order",
                "__ratio_binary",
            ]
        )
        .reset_index(drop=True)
    )

    actual_attack_fraction = float(binary_y(matched).mean())

    audit = {
        "available_rows": int(len(frame)),
        "available_attack": available_attack,
        "available_normal": available_normal,
        "selected_rows": int(len(matched)),
        "selected_attack": int(selected_attack),
        "selected_normal": int(selected_normal),
        "target_attack_pct": 100.0 * target_attack_fraction,
        "target_normal_pct": 100.0 * target_normal_fraction,
        "actual_attack_pct": 100.0 * actual_attack_fraction,
        "actual_normal_pct": 100.0 * (1.0 - actual_attack_fraction),
    }

    if round(audit["actual_normal_pct"], 1) != round(
        audit["target_normal_pct"], 1
    ):
        raise AssertionError(
            "The matched test set does not round to the requested "
            f"normal prevalence: {audit}"
        )

    return matched, audit


def ratio_match_test_sets(
    test_sets,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
):
    matched_sets = {}
    audit_rows = []

    for offset, (test_source, test_frame) in enumerate(
        test_sets.items()
    ):
        matched, audit = match_binary_test_ratio(
            test_frame,
            target_attack_fraction=target_attack_fraction,
            random_state=random_state + 1_000 * offset,
        )
        matched["ratio_match"] = "19.5% normal / 80.5% attack"
        matched_sets[test_source] = matched
        audit_rows.append({
            "test_source": test_source,
            **audit,
        })

    return matched_sets, pd.DataFrame(audit_rows)



In [ ]:

exp4_external_pool = df_master.copy()
exp4_external_pool["_binary_split"] = binary_y(exp4_external_pool)


exp4_external_train_natural, exp4_external_test_natural = split_each_source(
    exp4_external_pool,
    stratify_column="_binary_split",
    test_size=EXTERNAL_TEST_SIZE,
    seed=PREPARATION_SEED,
)

exp4_external_train_natural = (
    exp4_external_train_natural
    .drop(columns="_binary_split")
)
exp4_external_test_natural = (
    exp4_external_test_natural
    .drop(columns="_binary_split")
)


exp4_external_train_balanced = balance_binary_external_training(
    exp4_external_train_natural,
    seed=PREPARATION_SEED,
)

exp4_train_natural = pd.concat(
    [train_df, exp4_external_train_natural],
    ignore_index=True,
    sort=False,
)

exp4_train_balanced = pd.concat(
    [train_df, exp4_external_train_balanced],
    ignore_index=True,
    sort=False,
)

exp4_natural_tests = {
    "original": test_df.reset_index(drop=True),
    **{
        source: part.reset_index(drop=True)
        for source, part in exp4_external_test_natural.groupby(
            SOURCE_COLUMN,
            sort=True,
        )
    },
}

exp4_tests, exp4_test_ratio_manifest = ratio_match_test_sets(
    exp4_natural_tests,
    target_attack_fraction=TARGET_TEST_ATTACK_FRACTION,
    random_state=PREPARATION_SEED,
)

print("Experiment 4 natural train:", exp4_train_natural.shape)
print("Experiment 4 balanced train:", exp4_train_balanced.shape)
display(exp4_test_ratio_manifest)

for source, frame in exp4_tests.items():
    y_source = binary_y(frame)
    print(
        "Experiment 4 ratio-matched test",
        source,
        frame.shape,
        f"normal={(y_source == 0).mean() * 100:.3f}%",
        f"attack={(y_source == 1).mean() * 100:.3f}%",
    )


Experiment 4 natural train: (2513414, 48)
Experiment 4 balanced train: (559141, 48)


,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,311029,250436,60593,310733,250140,60593,80.5,19.5,80.499979,19.500021
1,cic2017,246899,48527,198372,60281,48526,11755,80.5,19.5,80.499660,19.500340
2,cic2018,208936,10564,198372,13122,10563,2559,80.5,19.5,80.498400,19.501600
3,darpa,323597,7392,316205,9182,7392,1790,80.5,19.5,80.505337,19.494663
4,unsw-nb15,86024,3489,82535,4334,3489,845,80.5,19.5,80.503000,19.497000


Experiment 4 ratio-matched test original (310733, 44) normal=19.500% attack=80.500%
Experiment 4 ratio-matched test cic2017 (60281, 49) normal=19.500% attack=80.500%
Experiment 4 ratio-matched test cic2018 (13122, 49) normal=19.502% attack=80.498%
Experiment 4 ratio-matched test darpa (9182, 49) normal=19.495% attack=80.505%
Experiment 4 ratio-matched test unsw-nb15 (4334, 49) normal=19.497% attack=80.503%


In [22]:
# Experiment 4 preparation audit

exp4_split_manifest = pd.DataFrame([
    {
        "dataset": "external_train_natural",
        "rows": len(exp4_external_train_natural),
    },
    {
        "dataset": "external_train_balanced",
        "rows": len(exp4_external_train_balanced),
    },
    {
        "dataset": "combined_train_natural",
        "rows": len(exp4_train_natural),
    },
    {
        "dataset": "combined_train_balanced",
        "rows": len(exp4_train_balanced),
    },
    {
        "dataset": "external_test_natural",
        "rows": len(exp4_external_test_natural),
    },
    {
        "dataset": "ratio_matched_test_rows",
        "rows": sum(len(frame) for frame in exp4_tests.values()),
    },
])

exp4_split_manifest.to_csv(
    RESULTS_DIR / "exp4_ratio19p5_normal_split_manifest.csv",
    index=False,
)
exp4_test_ratio_manifest.to_csv(
    RESULTS_DIR / "exp4_ratio19p5_normal_test_ratio_manifest.csv",
    index=False,
)

if SAVE_RATIO_MATCHED_TEST_SETS:
    for test_source, test_frame in exp4_tests.items():
        test_frame.to_csv(
            RESULTS_DIR
            / f"exp4_ratio19p5_normal_test_{test_source}.csv",
            index=False,
        )

display(exp4_split_manifest)
display(exp4_test_ratio_manifest)


,dataset,rows
0,external_train_natural,2019393
1,external_train_balanced,65120
2,combined_train_natural,2513414
3,combined_train_balanced,559141
4,external_test_natural,865456
5,ratio_matched_test_rows,397652


,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,311029,250436,60593,310733,250140,60593,80.5,19.5,80.499979,19.500021
1,cic2017,246899,48527,198372,60281,48526,11755,80.5,19.5,80.499660,19.500340
2,cic2018,208936,10564,198372,13122,10563,2559,80.5,19.5,80.498400,19.501600
3,darpa,323597,7392,316205,9182,7392,1790,80.5,19.5,80.505337,19.494663
4,unsw-nb15,86024,3489,82535,4334,3489,845,80.5,19.5,80.503000,19.497000


In [23]:
def calculate_binary_metrics(y_true, y_pred, y_score=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    result = {
        "n_test": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) else np.nan
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, y_score)
    else:
        result["roc_auc"] = np.nan

    return result


def category_recall_table(frame, y_true, y_pred):
    labels = (
        frame[LABEL_COLUMN]
        .astype(str)
        .str.rstrip(".")
        .to_numpy()
    )

    rows = []
    for category in sorted(pd.unique(labels)):
        mask = labels == category
        category_y = y_true[mask]
        category_prediction = y_pred[mask]

        rows.append({
            "test_category": category,
            "support": int(mask.sum()),
            "binary_recall": (
                recall_score(
                    category_y,
                    category_prediction,
                    pos_label=1,
                    zero_division=0,
                )
                if np.any(category_y == 1)
                else np.nan
            ),
            "benign_specificity": (
                recall_score(
                    category_y,
                    category_prediction,
                    pos_label=0,
                    zero_division=0,
                )
                if np.any(category_y == 0)
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def fit_ordinal_minmax(train_frame):
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    numerical_imputer = SimpleImputer(strategy="median")

    categorical = categorical_imputer.fit_transform(
        train_frame[CATEGORICAL_COLUMNS]
    )
    numerical = numerical_imputer.fit_transform(
        train_frame[NUMERICAL_COLUMNS]
    )

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32,
    )
    categorical = encoder.fit_transform(categorical)

    assembled = pd.DataFrame(index=train_frame.index)
    for index, column in enumerate(CATEGORICAL_COLUMNS):
        assembled[column] = categorical[:, index]
    for index, column in enumerate(NUMERICAL_COLUMNS):
        assembled[column] = numerical[:, index]

    scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
    X = scaler.fit_transform(
        assembled[FEATURE_COLUMNS]
    ).astype("float32")

    return {
        "categorical_imputer": categorical_imputer,
        "numerical_imputer": numerical_imputer,
        "encoder": encoder,
        "scaler": scaler,
    }, X


def transform_ordinal_minmax(preprocessor, frame):
    categorical = preprocessor[
        "categorical_imputer"
    ].transform(frame[CATEGORICAL_COLUMNS])
    numerical = preprocessor[
        "numerical_imputer"
    ].transform(frame[NUMERICAL_COLUMNS])

    categorical = preprocessor["encoder"].transform(categorical)

    assembled = pd.DataFrame(index=frame.index)
    for index, column in enumerate(CATEGORICAL_COLUMNS):
        assembled[column] = categorical[:, index]
    for index, column in enumerate(NUMERICAL_COLUMNS):
        assembled[column] = numerical[:, index]

    return preprocessor["scaler"].transform(
        assembled[FEATURE_COLUMNS]
    ).astype("float32")


In [24]:
XIAO_PROTOCOL_CATEGORIES = ["icmp", "tcp", "udp"]

XIAO_SERVICE_CATEGORIES = [
    "aol", "auth", "bgp", "courier", "csnet_ns", "ctf",
    "daytime", "discard", "domain", "domain_u", "echo",
    "eco_i", "ecr_i", "efs", "exec", "finger", "ftp",
    "ftp_data", "gopher", "harvest", "hostnames", "http",
    "http_2784", "http_443", "http_8001", "imap4", "irc",
    "iso_tsap", "klogin", "kshell", "ldap", "link", "login",
    "mtp", "name", "netbios_dgm", "netbios_ns",
    "netbios_ssn", "netstat", "nnsp", "nntp", "ntp_u",
    "other", "pm_dump", "pop_2", "pop_3", "printer",
    "private", "red_i", "remote_job", "rje", "shell",
    "smtp", "sql_net", "ssh", "sunrpc", "supdup", "systat",
    "telnet", "tftp_u", "tim_i", "time", "urh_i", "urp_i",
    "uucp", "uucp_path", "vmnet", "whois", "x11", "z39_50",
]

XIAO_FLAG_CATEGORIES = [
    "oth", "rej", "rsto", "rstos0", "rstr",
    "s0", "s1", "s2", "s3", "sf", "sh",
]


def _xiao_clean_frame(frame):
    cleaned = frame[FEATURE_COLUMNS].copy()

    for column in CATEGORICAL_COLUMNS:
        cleaned[column] = (
            cleaned[column]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    for column in NUMERICAL_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column],
            errors="coerce",
        )

    return cleaned


def build_xiao_cnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(10, 10, 1)),
        tf.keras.layers.Conv2D(
            8, (2, 2), padding="same",
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            bias_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2), strides=(1, 1)
        ),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Conv2D(
            16, (2, 2), padding="same",
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            bias_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
        ),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2), strides=(1, 1)
        ),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            64,
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            bias_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
        ),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(2, activation="softmax"),
    ])


def fit_xiao_ae_cnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    X_train_df = _xiao_clean_frame(train_frame)

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                MinMaxScaler(
                    feature_range=(0, 1),
                    clip=True,
                ),
                NUMERICAL_COLUMNS,
            ),
            (
                "categorical",
                OneHotEncoder(
                    categories=[
                        XIAO_PROTOCOL_CATEGORIES,
                        XIAO_SERVICE_CATEGORIES,
                        XIAO_FLAG_CATEGORIES,
                    ],
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float32,
                ),
                CATEGORICAL_COLUMNS,
            ),
        ],
        sparse_threshold=0.0,
    )

    print("  Xiao: fitting preprocessing...", flush=True)
    X_train = preprocessor.fit_transform(
        X_train_df
    ).astype("float32")

    if X_train.shape[1] != 122:
        raise ValueError(
            f"Xiao pipeline expected 122 processed features, "
            f"received {X_train.shape[1]}."
        )

    autoencoder = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(122,)),
        tf.keras.layers.Dense(
            100,
            activation="relu",
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            bias_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            name="bottleneck_100",
        ),
        tf.keras.layers.Dense(
            122,
            activation="sigmoid",
            kernel_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
            bias_initializer=tf.keras.initializers.RandomNormal(
                mean=0.0, stddev=0.05
            ),
        ),
    ])

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.MeanSquaredError(),
    )

    print("  Xiao: training autoencoder...", flush=True)
    autoencoder.fit(
        X_train,
        X_train,
        epochs=EPOCHS["xiao_autoencoder"],
        batch_size=BATCH_SIZES["xiao"],
        shuffle=True,
        verbose=2,
    )

    encoder = tf.keras.Model(
        inputs=autoencoder.inputs,
        outputs=autoencoder.get_layer(
            "bottleneck_100"
        ).output,
    )

    latent = encoder.predict(
        X_train,
        batch_size=1024,
        verbose=1,
    ).astype("float32")
    latent = latent.reshape(-1, 10, 10, 1)

    model = build_xiao_cnn()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    print("  Xiao: training CNN...", flush=True)
    model.fit(
        latent,
        binary_y(train_frame),
        epochs=EPOCHS["xiao_cnn"],
        batch_size=BATCH_SIZES["xiao"],
        shuffle=True,
        verbose=2,
    )

    return {
        "preprocessor": preprocessor,
        "encoder": encoder,
        "model": model,
        "seed": seed,
    }


def predict_xiao_ae_cnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_test = fitted["preprocessor"].transform(
        _xiao_clean_frame(evaluated)
    ).astype("float32")

    latent = fitted["encoder"].predict(
        X_test,
        batch_size=1024,
        verbose=0,
    ).astype("float32").reshape(-1, 10, 10, 1)

    probabilities = fitted["model"].predict(
        latent,
        batch_size=1024,
        verbose=0,
    )
    prediction = np.argmax(
        probabilities,
        axis=1,
    ).astype("int8")

    return (
        binary_y(evaluated),
        prediction,
        probabilities[:, 1],
        evaluated,
    )


In [25]:
JIA_PROTOCOL_MAPPING = {
    "icmp": 1, "tcp": 2, "udp": 3,
}

JIA_SERVICE_MAPPING = {
    "domain_u": 1, "domain-u": 1, "ecr_i": 2, "eco_i": 3,
    "finger": 4, "ftp_data": 5, "ftp": 6, "http": 7,
    "hostnames": 8, "hostname": 8, "imap": 9, "login": 10,
    "mtp": 11, "netstat": 12, "other": 13, "private": 14,
    "smtp": 15, "systat": 16, "telnet": 17, "time": 18,
    "uucp": 19,
}

JIA_FLAG_MAPPING = {
    "REJ": 1, "RSTO": 2, "RSTR": 3, "S0": 4,
    "S3": 5, "SF": 6, "SH": 7,
}


def transform_jia_raw(frame):
    result = frame[FEATURE_COLUMNS].copy()

    result["protocol_type"] = (
        result["protocol_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(JIA_PROTOCOL_MAPPING)
        .fillna(4)
    )

    result["service"] = (
        result["service"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(JIA_SERVICE_MAPPING)
        .fillna(20)
    )

    result["flag"] = (
        result["flag"]
        .astype(str)
        .str.strip()
        .str.upper()
        .map(JIA_FLAG_MAPPING)
        .fillna(8)
    )

    for column in FEATURE_COLUMNS:
        result[column] = pd.to_numeric(
            result[column],
            errors="coerce",
        )

    return result


def build_jia_dnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(41,)),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-8,
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def fit_jia_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    X_raw = transform_jia_raw(train_frame)

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X_raw)

    scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
    X_train = scaler.fit_transform(
        X_imputed
    ).astype("float32")

    model = build_jia_dnn()
    print("  Jia: training DNN...", flush=True)
    model.fit(
        X_train,
        binary_y(train_frame),
        epochs=EPOCHS["jia_dnn"],
        batch_size=BATCH_SIZES["jia"],
        shuffle=True,
        verbose=2,
    )

    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "seed": seed,
    }


def predict_jia_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_raw = transform_jia_raw(evaluated)
    X_test = fitted["imputer"].transform(X_raw)
    X_test = fitted["scaler"].transform(
        X_test
    ).astype("float32")

    probability = fitted["model"].predict(
        X_test,
        batch_size=1024,
        verbose=0,
    ).reshape(-1)
    prediction = (probability >= 0.5).astype("int8")

    return binary_y(evaluated), prediction, probability, evaluated


In [26]:
def fit_anuja_preprocessor(train_part):
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    numerical_imputer = SimpleImputer(strategy="median")

    categorical = categorical_imputer.fit_transform(
        train_part[CATEGORICAL_COLUMNS]
    )
    numerical = numerical_imputer.fit_transform(
        train_part[NUMERICAL_COLUMNS]
    )

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32,
    )
    categorical = encoder.fit_transform(categorical)

    scaler = StandardScaler()
    numerical = scaler.fit_transform(
        numerical
    ).astype("float32")

    X = np.empty(
        (len(train_part), len(FEATURE_COLUMNS)),
        dtype="float32",
    )

    categorical_indices = [
        FEATURE_COLUMNS.index(column)
        for column in CATEGORICAL_COLUMNS
    ]
    numerical_indices = [
        FEATURE_COLUMNS.index(column)
        for column in NUMERICAL_COLUMNS
    ]

    X[:, categorical_indices] = categorical
    X[:, numerical_indices] = numerical

    return {
        "categorical_imputer": categorical_imputer,
        "numerical_imputer": numerical_imputer,
        "encoder": encoder,
        "scaler": scaler,
        "categorical_indices": categorical_indices,
        "numerical_indices": numerical_indices,
    }, X


def transform_anuja(preprocessor, frame):
    categorical = preprocessor[
        "categorical_imputer"
    ].transform(frame[CATEGORICAL_COLUMNS])
    categorical = preprocessor[
        "encoder"
    ].transform(categorical)

    numerical = preprocessor[
        "numerical_imputer"
    ].transform(frame[NUMERICAL_COLUMNS])
    numerical = preprocessor[
        "scaler"
    ].transform(numerical).astype("float32")

    X = np.empty(
        (len(frame), len(FEATURE_COLUMNS)),
        dtype="float32",
    )
    X[:, preprocessor["categorical_indices"]] = categorical
    X[:, preprocessor["numerical_indices"]] = numerical
    return X


def build_anuja_autoencoder():
    autoencoder_input = tf.keras.layers.Input(shape=(41,))
    encoded_64 = tf.keras.layers.Dense(
        64, activation="relu"
    )(autoencoder_input)
    latent_32 = tf.keras.layers.Dense(
        32, activation="relu", name="latent_32"
    )(encoded_64)
    decoded_64 = tf.keras.layers.Dense(
        64, activation="relu"
    )(latent_32)
    reconstruction = tf.keras.layers.Dense(
        41, activation="sigmoid"
    )(decoded_64)

    autoencoder = tf.keras.Model(
        autoencoder_input,
        reconstruction,
    )
    encoder = tf.keras.Model(
        autoencoder_input,
        latent_32,
    )
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=tf.keras.losses.MeanSquaredError(),
    )
    return autoencoder, encoder


def build_anuja_classifier():
    classifier_input = tf.keras.layers.Input(
        shape=(32, 1)
    )
    x = tf.keras.layers.Conv1D(
        64, 3, activation="relu", padding="valid"
    )(classifier_input)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
    x = tf.keras.layers.LSTM(
        64, activation="tanh"
    )(x)
    x = tf.keras.layers.Dense(
        64, activation="relu"
    )(x)
    output = tf.keras.layers.Dense(
        1, activation="sigmoid"
    )(x)

    model = tf.keras.Model(classifier_input, output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )
    return model


def fit_anuja_ae_cnn_lstm(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=VALIDATION_SIZE,
        random_state=seed,
        stratify=binary_y(train_frame),
    )

    preprocessor, X_train = fit_anuja_preprocessor(
        train_part
    )
    X_validation = transform_anuja(
        preprocessor,
        validation_part,
    )

    y_train = binary_y(train_part)
    y_validation = binary_y(validation_part)

    autoencoder, encoder = build_anuja_autoencoder()

    normal_train = X_train[y_train == 0]
    normal_validation = X_validation[y_validation == 0]

    print("  Anuja: training normal-only autoencoder...", flush=True)
    autoencoder.fit(
        normal_train,
        normal_train,
        validation_data=(
            normal_validation,
            normal_validation,
        ),
        epochs=EPOCHS["anuja_autoencoder"],
        batch_size=BATCH_SIZES["anuja"],
        shuffle=True,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
                verbose=1,
            )
        ],
        verbose=2,
    )

    latent_train = encoder.predict(
        X_train, batch_size=1024, verbose=0
    ).reshape(-1, 32, 1)
    latent_validation = encoder.predict(
        X_validation, batch_size=1024, verbose=0
    ).reshape(-1, 32, 1)

    classifier = build_anuja_classifier()

    print("  Anuja: training CNN-LSTM...", flush=True)
    classifier.fit(
        latent_train,
        y_train,
        validation_data=(
            latent_validation,
            y_validation,
        ),
        epochs=EPOCHS["anuja_classifier"],
        batch_size=BATCH_SIZES["anuja"],
        shuffle=True,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
                verbose=1,
            )
        ],
        verbose=2,
    )

    validation_probability = classifier.predict(
        latent_validation,
        batch_size=1024,
        verbose=0,
    ).reshape(-1)

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_validation,
            validation_probability,
        )
    )

    f1_values = (
        2
        * precision_values[:-1]
        * recall_values[:-1]
        / (
            precision_values[:-1]
            + recall_values[:-1]
            + 1e-12
        )
    )

    threshold = (
        float(thresholds[np.argmax(f1_values)])
        if len(thresholds)
        else 0.5
    )

    print(
        f"  Anuja: validation-calibrated threshold={threshold:.6f}",
        flush=True,
    )

    return {
        "preprocessor": preprocessor,
        "encoder": encoder,
        "model": classifier,
        "threshold": threshold,
        "seed": seed,
    }


def predict_anuja_ae_cnn_lstm(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_test = transform_anuja(
        fitted["preprocessor"],
        evaluated,
    )
    latent = fitted["encoder"].predict(
        X_test,
        batch_size=1024,
        verbose=0,
    ).reshape(-1, 32, 1)

    probability = fitted["model"].predict(
        latent,
        batch_size=1024,
        verbose=0,
    ).reshape(-1)
    prediction = (
        probability >= fitted["threshold"]
    ).astype("int8")

    return binary_y(evaluated), prediction, probability, evaluated


In [27]:
def fit_smart_city_voting_ensemble(train_frame, seed):
    set_all_seeds(seed)

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    # The reproduced pipeline removes exact duplicates.
    dedupe_columns = FEATURE_COLUMNS + [LABEL_COLUMN]
    train_frame = (
        train_frame
        .drop_duplicates(subset=dedupe_columns)
        .reset_index(drop=True)
    )

    preprocessor, X_train = fit_ordinal_minmax(
        train_frame
    )

    # Preserve the original paper implementation:
    # 1 = normal, 0 = attack.
    y_normal = 1 - binary_y(train_frame)

    gnb = GaussianNB()
    tree = DecisionTreeClassifier(
        criterion="gini",
        splitter="best",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=seed,
    )
    xgb = XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        min_child_weight=1,
        gamma=0.0,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_alpha=0.0,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
    )

    print("  Ensemble: training GaussianNB...", flush=True)
    gnb.fit(X_train, y_normal)

    print("  Ensemble: training Decision Tree...", flush=True)
    tree.fit(X_train, y_normal)

    print("  Ensemble: training XGBoost...", flush=True)
    xgb.fit(X_train, y_normal)

    return {
        "preprocessor": preprocessor,
        "gnb": gnb,
        "tree": tree,
        "xgb": xgb,
        "seed": seed,
    }


def predict_smart_city_voting_ensemble(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted["seed"])
    X_test = transform_ordinal_minmax(
        fitted["preprocessor"],
        evaluated,
    )

    normal_votes = np.column_stack([
        fitted["gnb"].predict(X_test),
        fitted["tree"].predict(X_test),
        fitted["xgb"].predict(X_test),
    ])

    normal_prediction = (
        normal_votes.sum(axis=1) >= 2
    ).astype("int8")

    # Convert back to the common convention:
    # 0 = benign, 1 = attack.
    attack_prediction = 1 - normal_prediction

    # The reproduced ensemble score is the mean of hard votes.
    attack_score = 1.0 - normal_votes.mean(axis=1)

    return (
        binary_y(evaluated),
        attack_prediction,
        attack_score,
        evaluated,
    )


In [28]:
# Model registry

MODEL_REGISTRY = {
    "xiao_ae_cnn": {
        "fit": fit_xiao_ae_cnn,
        "predict": predict_xiao_ae_cnn,
    },
    "jia_dnn": {
        "fit": fit_jia_dnn,
        "predict": predict_jia_dnn,
    },
    "anuja_ae_cnn_lstm": {
        "fit": fit_anuja_ae_cnn_lstm,
        "predict": predict_anuja_ae_cnn_lstm,
    },
    "smart_city_voting_ensemble": {
        "fit": fit_smart_city_voting_ensemble,
        "predict": predict_smart_city_voting_ensemble,
    },
}

unknown_models = sorted(
    set(MODEL_KEYS) - set(MODEL_REGISTRY)
)
if unknown_models:
    raise KeyError(f"Unknown MODEL_KEYS: {unknown_models}")

print("Enabled models:", MODEL_KEYS)


Enabled models: ['xiao_ae_cnn', 'jia_dnn', 'anuja_ae_cnn_lstm', 'smart_city_voting_ensemble']


In [29]:
def _merge_and_save(path, new_frame, duplicate_columns):
    if path.exists():
        previous = pd.read_csv(path)
        combined = pd.concat(
            [previous, new_frame],
            ignore_index=True,
        )
    else:
        combined = new_frame.copy()

    combined = combined.drop_duplicates(
        subset=duplicate_columns,
        keep="last",
    )
    combined.to_csv(path, index=False)
    return combined


def evaluate_model_on_test_sets(
    model_key,
    train_frame,
    test_sets,
    experiment,
    training_variant,
    seed,
):
    adapter = MODEL_REGISTRY[model_key]

    print(
        f"\n{experiment} | {model_key} | "
        f"{training_variant} | seed={seed}",
        flush=True,
    )
    print("  Training once...", flush=True)

    fit_started = time.perf_counter()
    fitted = adapter["fit"](train_frame, seed)
    fit_seconds = time.perf_counter() - fit_started

    print(
        f"  Training completed in "
        f"{fit_seconds / 60:.2f} minutes.",
        flush=True,
    )

    overall_rows = []
    category_parts = []

    pooled_true = []
    pooled_prediction = []
    pooled_score = []

    for test_source, test_frame in test_sets.items():
        print(
            f"  Testing {test_source} "
            f"({len(test_frame):,} raw rows)...",
            flush=True,
        )

        prediction_started = time.perf_counter()
        (
            y_true,
            y_pred,
            y_score,
            evaluated_test,
        ) = adapter["predict"](fitted, test_frame)
        prediction_seconds = (
            time.perf_counter() - prediction_started
        )

        metrics = calculate_binary_metrics(
            y_true,
            y_pred,
            y_score,
        )

        evaluated_attack_fraction = float(np.mean(y_true == 1))

        overall_rows.append({
            "experiment": experiment,
            "training_variant": training_variant,
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            "n_train_raw": int(len(train_frame)),
            "fit_seconds": fit_seconds,
            "prediction_seconds": prediction_seconds,
            "test_attack_pct": 100.0 * evaluated_attack_fraction,
            "test_normal_pct": 100.0 * (1.0 - evaluated_attack_fraction),
            **metrics,
        })

        categories = category_recall_table(
            evaluated_test,
            y_true,
            y_pred,
        )
        categories["experiment"] = experiment
        categories["training_variant"] = training_variant
        categories["model"] = model_key
        categories["seed"] = seed
        categories["test_source"] = test_source
        category_parts.append(categories)

        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)

        print(
            f"    evaluated={len(y_true):,}, "
            f"accuracy={metrics['accuracy']:.4f}, "
            f"recall={metrics['recall']:.4f}, "
            f"f1={metrics['f1']:.4f}",
            flush=True,
        )

    pooled_true = np.concatenate(pooled_true)
    pooled_prediction = np.concatenate(pooled_prediction)
    pooled_score = np.concatenate(pooled_score)

    pooled_metrics = calculate_binary_metrics(
        pooled_true,
        pooled_prediction,
        pooled_score,
    )

    pooled_attack_fraction = float(np.mean(pooled_true == 1))

    overall_rows.append({
        "experiment": experiment,
        "training_variant": training_variant,
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "n_train_raw": int(len(train_frame)),
        "fit_seconds": fit_seconds,
        "prediction_seconds": sum(
            row["prediction_seconds"]
            for row in overall_rows
        ),
        "test_attack_pct": 100.0 * pooled_attack_fraction,
        "test_normal_pct": 100.0 * (1.0 - pooled_attack_fraction),
        **pooled_metrics,
    })

    del fitted
    gc.collect()
    tf.keras.backend.clear_session()

    return (
        pd.DataFrame(overall_rows),
        pd.concat(category_parts, ignore_index=True),
    )


def run_experiment(
    experiment,
    train_variants,
    test_sets,
):
    overall_path = (
        RESULTS_DIR
        / f"{experiment}_all_runs_partial.csv"
    )
    category_path = (
        RESULTS_DIR
        / f"{experiment}_category_runs_partial.csv"
    )

    completed = (
        pd.read_csv(overall_path)
        if overall_path.exists()
        else pd.DataFrame()
    )

    overall_accumulated = completed.copy()
    category_accumulated = (
        pd.read_csv(category_path)
        if category_path.exists()
        else pd.DataFrame()
    )

    expected_sources = set(test_sets) | {
        "POOLED_ALL_ROWS"
    }

    for model_key in MODEL_KEYS:
        for training_variant, train_frame in train_variants.items():
            for seed in SEEDS:
                if not completed.empty:
                    mask = (
                        completed["experiment"].eq(experiment)
                        & completed["model"].eq(model_key)
                        & completed[
                            "training_variant"
                        ].eq(training_variant)
                        & completed["seed"].eq(seed)
                    )
                    completed_sources = set(
                        completed.loc[
                            mask,
                            "test_source",
                        ].astype(str)
                    )
                else:
                    completed_sources = set()

                if expected_sources.issubset(
                    completed_sources
                ):
                    print(
                        f"SKIP completed: {model_key}, "
                        f"{training_variant}, seed={seed}",
                        flush=True,
                    )
                    continue

                overall, categories = (
                    evaluate_model_on_test_sets(
                        model_key=model_key,
                        train_frame=train_frame,
                        test_sets=test_sets,
                        experiment=experiment,
                        training_variant=training_variant,
                        seed=seed,
                    )
                )

                overall_accumulated = _merge_and_save(
                    overall_path,
                    overall,
                    [
                        "experiment",
                        "training_variant",
                        "model",
                        "seed",
                        "test_source",
                    ],
                )

                category_accumulated = _merge_and_save(
                    category_path,
                    categories,
                    [
                        "experiment",
                        "training_variant",
                        "model",
                        "seed",
                        "test_source",
                        "test_category",
                    ],
                )

                completed = overall_accumulated

    return overall_accumulated, category_accumulated


In [30]:
# Final result summaries and exports

def summarize_overall(results):
    metrics = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "macro_f1",
        "mcc",
        "false_positive_rate",
        "roc_auc",
    ]

    return (
        results
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "test_source",
            ],
            dropna=False,
        )[metrics]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


def summarize_categories(results):
    return (
        results
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "test_source",
                "test_category",
            ],
            dropna=False,
        )
        .agg(
            support=("support", "first"),
            binary_recall_mean=(
                "binary_recall", "mean"
            ),
            binary_recall_std=(
                "binary_recall", "std"
            ),
            benign_specificity_mean=(
                "benign_specificity", "mean"
            ),
            benign_specificity_std=(
                "benign_specificity", "std"
            ),
        )
        .reset_index()
    )


def source_macro_summary(results):
    source_only = results[
        results["test_source"].ne(
            "POOLED_ALL_ROWS"
        )
    ]

    by_seed = (
        source_only
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
                "seed",
            ]
        )
        .agg(
            source_macro_accuracy=(
                "accuracy", "mean"
            ),
            source_macro_balanced_accuracy=(
                "balanced_accuracy", "mean"
            ),
            source_macro_f1=("f1", "mean"),
            source_macro_mcc=("mcc", "mean"),
            source_macro_recall=("recall", "mean"),
        )
        .reset_index()
    )

    return (
        by_seed
        .groupby(
            [
                "experiment",
                "training_variant",
                "model",
            ]
        )
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


def save_final_results(
    experiment,
    overall,
    categories,
):
    overall_summary = summarize_overall(overall)
    category_summary = summarize_categories(categories)
    source_summary = source_macro_summary(overall)

    overall.to_csv(
        RESULTS_DIR / f"{experiment}_all_runs.csv",
        index=False,
    )
    categories.to_csv(
        RESULTS_DIR / f"{experiment}_category_runs.csv",
        index=False,
    )
    overall_summary.to_csv(
        RESULTS_DIR / f"{experiment}_summary.csv",
        index=False,
    )
    category_summary.to_csv(
        RESULTS_DIR / f"{experiment}_category_summary.csv",
        index=False,
    )
    source_summary.to_csv(
        RESULTS_DIR / f"{experiment}_source_macro_summary.csv",
        index=False,
    )

    return (
        overall_summary,
        category_summary,
        source_summary,
    )


In [31]:
# Experiment 4 — mixed-source adaptation with 19.5% normal test prevalence

exp4_overall, exp4_categories = run_experiment(
    experiment=EXP4_EXPERIMENT,
    train_variants={
        "mixed_external_balanced": exp4_train_balanced,
        # "mixed_external_natural": exp4_train_natural,
    },
    test_sets=exp4_tests,
)

(
    exp4_summary,
    exp4_category_summary,
    exp4_source_macro_summary,
) = save_final_results(
    EXP4_EXPERIMENT,
    exp4_overall,
    exp4_categories,
)

exp4_test_ratio_manifest.to_csv(
    RESULTS_DIR
    / f"{EXP4_EXPERIMENT}_test_ratio_manifest.csv",
    index=False,
)

print(
    "Primary row-weighted mixed-test result:",
    "test_source == 'POOLED_ALL_ROWS'",
)
print(
    "Equal-source result:",
    f"{EXP4_EXPERIMENT}_source_macro_summary.csv",
)

display(exp4_test_ratio_manifest)
display(exp4_summary)
display(exp4_category_summary)
display(exp4_source_macro_summary)



exp4_mixed_traffic_test_ratio19p5_normal | xiao_ae_cnn | mixed_external_balanced | seed=11
  Training once...
  Xiao: fitting preprocessing...
  Xiao: training autoencoder...
Epoch 1/20
4369/4369 - 22s - 5ms/step - loss: 0.0031
Epoch 2/20
4369/4369 - 7s - 2ms/step - loss: 1.8417e-04
Epoch 3/20
4369/4369 - 7s - 2ms/step - loss: 1.1143e-04
Epoch 4/20
4369/4369 - 7s - 2ms/step - loss: 5.8805e-05
Epoch 5/20
4369/4369 - 7s - 2ms/step - loss: 2.9215e-05
Epoch 6/20
4369/4369 - 7s - 2ms/step - loss: 2.3489e-05
Epoch 7/20
4369/4369 - 7s - 2ms/step - loss: 2.1311e-05
Epoch 8/20
4369/4369 - 7s - 2ms/step - loss: 1.9610e-05
Epoch 9/20
4369/4369 - 7s - 2ms/step - loss: 1.7990e-05
Epoch 10/20
4369/4369 - 7s - 2ms/step - loss: 1.6670e-05
Epoch 11/20
4369/4369 - 7s - 2ms/step - loss: 1.5689e-05
Epoch 12/20
4369/4369 - 7s - 2ms/step - loss: 1.4917e-05
Epoch 13/20
4369/4369 - 7s - 2ms/step - loss: 1.4253e-05
Epoch 14/20
4369/4369 - 7s - 2ms/step - loss: 1.3673e-05
Epoch 15/20
4369/4369 - 7s - 2ms/step 

,test_source,available_rows,available_attack,available_normal,selected_rows,selected_attack,selected_normal,target_attack_pct,target_normal_pct,actual_attack_pct,actual_normal_pct
0,original,311029,250436,60593,310733,250140,60593,80.5,19.5,80.499979,19.500021
1,cic2017,246899,48527,198372,60281,48526,11755,80.5,19.5,80.499660,19.500340
2,cic2018,208936,10564,198372,13122,10563,2559,80.5,19.5,80.498400,19.501600
3,darpa,323597,7392,316205,9182,7392,1790,80.5,19.5,80.505337,19.494663
4,unsw-nb15,86024,3489,82535,4334,3489,845,80.5,19.5,80.503000,19.497000


experiment         training_variant  \
                                                                        
0   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
1   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
2   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
3   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
4   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
5   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
6   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
7   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
8   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
9   exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
10  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
11  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
12  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
13  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
14  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
15  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
16  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
17  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
18  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
19  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
20  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
21  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
22  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
23  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   

                         model      test_source  accuracy                      \
                                                     mean       std       min   
0            anuja_ae_cnn_lstm  POOLED_ALL_ROWS  0.936435  0.000391  0.936014   
1            anuja_ae_cnn_lstm          cic2017  0.978965  0.005211  0.971202   
2            anuja_ae_cnn_lstm          cic2018  0.994483  0.002100  0.990855   
3            anuja_ae_cnn_lstm            darpa  0.949074  0.019008  0.920170   
4            anuja_ae_cnn_lstm         original  0.926571  0.001439  0.925064   
5            anuja_ae_cnn_lstm        unsw-nb15  0.849608  0.009851  0.839179   
6                      jia_dnn  POOLED_ALL_ROWS  0.938549  0.001174  0.937149   
7                      jia_dnn          cic2017  0.987704  0.000625  0.987061   
8                      jia_dnn          cic2018  0.995793  0.001254  0.994361   
9                      jia_dnn            darpa  0.973731  0.002711  0.970268   
10                     jia_dnn         original  0.926180  0.001575  0.924298   
11                     jia_dnn        unsw-nb15  0.893816  0.016433  0.867790   
12  smart_city_voting_ensemble  POOLED_ALL_ROWS  0.941011  0.000293  0.940647   
13  smart_city_voting_ensemble          cic2017  0.995607  0.000112  0.995488   
14  smart_city_voting_ensemble          cic2018  0.998674  0.000068  0.998628   
15  smart_city_voting_ensemble            darpa  0.989545  0.000133  0.989327   
16  smart_city_voting_ensemble         original  0.926238  0.000368  0.925792   
17  smart_city_voting_ensemble        unsw-nb15  0.963406  0.001865  0.960545   
18                 xiao_ae_cnn  POOLED_ALL_ROWS  0.940444  0.001091  0.939472   
19                 xiao_ae_cnn          cic2017  0.986901  0.001019  0.985402   
20                 xiao_ae_cnn          cic2018  0.996052  0.000789  0.995275   
21                 xiao_ae_cnn            darpa  0.972163  0.000788  0.971030   
22                 xiao_ae_cnn         original  0.928642  0.001372  0.927639   
23                 xiao_ae_cnn        unsw-nb15  0.904938  0.011040  0.892709   

             balanced_accura

,experiment,training_variant,model,test_source,test_category,support,binary_recall_mean,binary_recall_std,benign_specificity_mean,benign_specificity_std
0,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,anuja_ae_cnn_lstm,cic2017,Infiltration,9,0.155556,0.126686,NaN,NaN
1,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,anuja_ae_cnn_lstm,cic2017,Web Attack - Brute Force,340,0.085882,0.047671,NaN,NaN
2,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,anuja_ae_cnn_lstm,cic2017,Web Attack - Sql Injection,1,0.200000,0.447214,NaN,NaN
3,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,anuja_ae_cnn_lstm,cic2017,Web Attack - XSS,175,0.062857,0.051429,NaN,NaN
4,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,anuja_ae_cnn_lstm,cic2017,benign,11755,NaN,NaN,0.970906,0.008146
...,...,...,...,...,...,...,...,...,...,...
247,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,xiao_ae_cnn,unsw-nb15,Shellcode,52,0.930769,0.010533,NaN,NaN
248,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,xiao_ae_cnn,unsw-nb15,Worms,9,1.000000,0.000000,NaN,NaN
249,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,xiao_ae_cnn,unsw-nb15,benign,845,NaN,NaN,0.915976,0.009281
250,exp4_mixed_traffic_test_ratio19p5_normal,mixed_external_balanced,xiao_ae_cnn,unsw-nb15,dos,213,0.894836,0.031946,NaN,NaN


experiment         training_variant  \
                                                                       
0  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
1  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
2  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   
3  exp4_mixed_traffic_test_ratio19p5_normal  mixed_external_balanced   

                        model  seed                    source_macro_accuracy  \
                               mean        std min max                  mean   
0           anuja_ae_cnn_lstm  33.0  17.392527  11  55              0.939740   
1                     jia_dnn  33.0  17.392527  11  55              0.955445   
2  smart_city_voting_ensemble  33.0  17.392527  11  55              0.974694   
3                 xiao_ae_cnn  33.0  17.392527  11  55              0.957739   

                       ... source_macro_f1           source_macro_mcc  \
        std       min  ...             min       max             mean   
0  0.002910  0.934840  ...        0.957428  0.961899         0.845366   
1  0.002523  0.951221  ...        0.968388  0.973127         0.877978   
2  0.000389  0.974139  ...        0.983430  0.984118         0.929896   
3  0.001929  0.956003  ...        0.971592  0.975024         0.883096   

                                source_macro_recall                      \
        std       min       max                mean       std       min   
0  0.008701  0.830773  0.854222            0.932986  0.002611  0.928714   
1  0.005257  0.868827  0.882056            0.954230  0.003525  0.949355   
2  0.001000  0.928449  0.931197            0.970982  0.000464  0.970305   
3  0.003130  0.880806  0.888467            0.956419  0.003660  0.952644   

             
        max  
0  0.934890  
1  0.958374  
2  0.971567  
3  0.961896  

[4 rows x 27 columns]